TASK 3: VALIDATE (validate.py)

Nhiệm vụ: Đọc taskValues từ 2 task trước, kiểm định các điều kiện bất biến nghiệp vụ. Nếu sai, lập tức raise Exception để dừng Job!

In [0]:
TEN = "duykh"
GOLD = f"workspace.gold_{TEN}.fact_daily_shop_sales_{TEN}"

# 1. Đọc toàn bộ các giá trị từ 2 task trước
bronze_n = dbutils.jobs.taskValues.get(taskKey="ingest", key="bronze_rows", debugValue=35)
silver_n = dbutils.jobs.taskValues.get(taskKey="transform", key="silver_rows", debugValue=33)
gold_n   = dbutils.jobs.taskValues.get(taskKey="transform", key="gold_rows", debugValue=28)

print("--- KIỂM TRA SỐ LIỆU ĐIỀU PHỐI ---")
print(f"Bronze Rows: {bronze_n}")
print(f"Silver Rows: {silver_n}")
print(f"Gold Rows:   {gold_n}")

errors = []

# Kiểm tra quy tắc 1: Silver không thể nhiều dòng hơn Bronze
if silver_n > bronze_n:
    errors.append(f"Quy tắc 1 vi phạm: Silver ({silver_n}) > Bronze ({bronze_n})!")

# Kiểm tra quy tắc 2: Nếu đã nạp đủ 5 file, Silver phải đạt 33 dòng chuẩn
if bronze_n == 35 and silver_n != 33:
    errors.append(f"Quy tắc 2 vi phạm: Với 35 dòng Bronze, Silver phải có đúng 33 dòng, thực tế={silver_n}!")

# Kiểm tra quy tắc 3: Doanh thu tổng của hệ thống không được âm hoặc bằng 0
total_rev = spark.sql(f"SELECT SUM(revenue) AS t FROM {GOLD}").collect()[0]["t"]
print(f"Tổng doanh thu Gold: {total_rev:,.0f} VNĐ")

if total_rev is None or total_rev <= 0:
    errors.append("Quy tắc 3 vi phạm: Doanh thu hệ thống bị NULL hoặc <= 0!")

# 4. Chốt chặn an toàn: Báo lỗi nếu có bất kỳ vi phạm nào
if len(errors) > 0:
    error_msg = " [CIRCUIT BREAKER KÍCH HOẠT] PHÁT HIỆN LỖI CHẤT LƯỢNG DỮ LIỆU:\n" + "\n".join(errors)
    print(error_msg)
    raise ValueError(error_msg)
else:
    print("TOÀN BỘ CÁC BÀI KIỂM THỬ DATA QUALITY ĐÃ ĐẠT!")
